# Case study: differentiating a LIBOR Market Model with Tangent

Interest-rate desks price caps, floors and swaptions by **Monte-Carlo
simulation of forward LIBOR rates**, then need *greeks* — the price's
sensitivity to every initial rate (deltas) and every volatility (vegas). The
traditional way to get them is **bump-and-revalue**: perturb one input, re-run
the whole simulation, difference. For `K` inputs that is `2K` full Monte-Carlo
revaluations, and each difference carries Monte-Carlo noise.

Automatic differentiation replaces all of that with **one reverse pass**. A
sibling project, [`kooderive`](https://github.com/), implements LMM-style Monte
Carlo in JAX; here we take the *same class of model written as plain NumPy
loops* and let **Tangent** differentiate it — turning the simulator into
readable gradient code you can inspect, verify, and edit.

What makes this a real test (not a toy): the spot-measure drift of each forward
rate depends on **all the other not-yet-reset rates**, so every time step
couples the whole curve. That state-dependent drift, wrapped in a Monte-Carlo
loop, is exactly the kind of "messy simulator" tracing frameworks were built
for — and Tangent differentiates it as-is.


In [1]:
import numpy as np
import tangent


## The simulator (plain NumPy)

A single-factor LIBOR Market Model under the spot measure, vectorized over
Monte-Carlo paths (`F` has shape `paths × rates`). The messy coupling — each
rate's drift is a prefix sum over the not-yet-reset rates — is expressed as one
matrix multiply against a constant lower-triangular indicator `L`
(`L[k, i] = 1` iff `k <= i`), which keeps the code both readable *and*
differentiable.

Tangent constraints shape the style: no in-place index assignment or `.copy()`,
and no `.shape` on a differentiated argument — so shapes come in as constants
and the first-rate selection and the ATM payoff use array arithmetic (a
selector vector `e0`, and a `(x > 0)` mask for `max(·, 0)`).

In [2]:
def caplet_price(F0, sigma, Z, K, tau, dt, L, e0, M, N, n_steps):
    """Stylized spot-measure LMM price of a caplet on the first forward rate.

    Differentiated w.r.t. F0 (initial forward curve -> deltas) and sigma
    (per-rate vols -> vegas). Z (pre-drawn normals), K, tau, dt, L, e0 and the
    integer shapes are constants.
    """
    sqrt_dt = dt ** 0.5
    F = F0 * np.ones((M, N))                     # broadcast the initial curve to every path
    for step in range(n_steps):
        g = tau * sigma * F / (1.0 + tau * F)    # per-rate drift contribution (paths x rates)
        drift = sigma * (g @ L) - 0.5 * sigma * sigma   # spot-measure drift couples the curve
        F = F * np.exp(drift * dt + sigma * (sqrt_dt * Z[step]))
    F_reset = F @ e0                             # first forward rate at its reset date
    intrinsic = F_reset - K
    payoff = tau * (intrinsic * (intrinsic > 0.0)) / (1.0 + tau * F_reset)
    return np.sum(payoff) / M                    # Monte-Carlo price


## Set up a market and price it

In [3]:
N, M, n_steps = 5, 20000, 8       # rates, MC paths, time steps
tau, K, T0 = 0.5, 0.03, 1.0
dt = T0 / n_steps

F0 = np.array([0.030, 0.032, 0.034, 0.035, 0.036])   # initial forward curve
sigma = np.array([0.20, 0.22, 0.24, 0.23, 0.21])     # per-rate volatilities

rng = np.random.RandomState(0)
Z = rng.standard_normal((n_steps, M, 1))   # common random numbers (fixed across bumps)
L = np.tril(np.ones((N, N)))               # L[k, i] = 1 iff k <= i
e0 = np.eye(N)[0]                          # selects the first forward rate

CONSTS = (Z, K, tau, dt, L, e0, M, N, n_steps)
price = caplet_price(F0, sigma, *CONSTS)
print("Monte-Carlo caplet price: %.6f" % price)


Monte-Carlo caplet price: 0.001217


## All greeks in one reverse pass

`tangent.grad(..., wrt=(0, 1))` differentiates the simulator with respect to
the whole initial curve **and** every volatility at once. One call returns all
five deltas and all five vegas — no per-input re-simulation.

In [4]:
dprice = tangent.grad(caplet_price, wrt=(0, 1))
delta, vega = dprice(F0, sigma, *CONSTS)
print("deltas dPrice/dF0   :", np.round(delta, 6))
print("vegas  dPrice/dsigma:", np.round(vega, 6))


deltas dPrice/dF0   : [2.72734e-01 1.90000e-04 2.08000e-04 1.98000e-04 1.80000e-04]
vegas  dPrice/dsigma: [6.082e-03 3.000e-05 3.300e-05 3.300e-05 3.400e-05]


## Validate against bump-and-revalue

The desk's traditional method: bump each input up and down, re-price, difference.
With common random numbers (the same `Z`) this is the honest check that the AD
greeks are correct. Note the caplet is struck at-the-money (`K = F0[0]`), so the
`max(·, 0)` payoff has a kink there — the pathwise delta is a subgradient and
finite differences are noisy at that exact point; vegas (smooth) agree to
machine precision.

In [5]:
h = 1e-6
fd_delta = np.zeros(N)
fd_vega = np.zeros(N)
for i in range(N):
    Fp = F0.copy(); Fp[i] += h
    Fm = F0.copy(); Fm[i] -= h
    fd_delta[i] = (caplet_price(Fp, sigma, *CONSTS) - caplet_price(Fm, sigma, *CONSTS)) / (2 * h)
    sp = sigma.copy(); sp[i] += h
    sm = sigma.copy(); sm[i] -= h
    fd_vega[i] = (caplet_price(F0, sp, *CONSTS) - caplet_price(F0, sm, *CONSTS)) / (2 * h)

print("AD delta:", np.round(delta, 6))
print("FD delta:", np.round(fd_delta, 6))
print("  max|AD - FD| =", np.max(np.abs(delta - fd_delta)))
print()
print("AD vega :", np.round(vega, 6))
print("FD vega :", np.round(fd_vega, 6))
print("  max|AD - FD| =", np.max(np.abs(vega - fd_vega)))


AD delta: [2.72734e-01 1.90000e-04 2.08000e-04 1.98000e-04 1.80000e-04]
FD delta: [2.72728e-01 1.90000e-04 2.08000e-04 1.98000e-04 1.80000e-04]
  max|AD - FD| = 6.7102115244854765e-06

AD vega : [6.082e-03 3.000e-05 3.300e-05 3.300e-05 3.400e-05]
FD vega : [6.082e-03 3.000e-05 3.300e-05 3.300e-05 3.400e-05]
  max|AD - FD| = 1.2864802810281378e-13


## Why AD wins: one pass vs. 2K revaluations

Bump-and-revalue costs two full simulations per input — here `2 x 10 = 20`
re-prices for the ten greeks. Reverse-mode AD produces **all ten in a single
backward pass**. The gap widens linearly with the number of sensitivities (real
books have hundreds).

In [6]:
import time

def timeit(fn, repeats=3):
    best = float("inf")
    for _ in range(repeats):
        t = time.perf_counter(); fn(); best = min(best, time.perf_counter() - t)
    return best

def all_greeks_ad():
    return dprice(F0, sigma, *CONSTS)

def all_greeks_fd():
    d = np.zeros(N); v = np.zeros(N)
    for i in range(N):
        Fp = F0.copy(); Fp[i] += h; Fm = F0.copy(); Fm[i] -= h
        d[i] = (caplet_price(Fp, sigma, *CONSTS) - caplet_price(Fm, sigma, *CONSTS)) / (2 * h)
        sp = sigma.copy(); sp[i] += h; sm = sigma.copy(); sm[i] -= h
        v[i] = (caplet_price(F0, sp, *CONSTS) - caplet_price(F0, sm, *CONSTS)) / (2 * h)
    return d, v

t_ad = timeit(all_greeks_ad)
t_fd = timeit(all_greeks_fd)
print("AD (1 reverse pass, all 10 greeks): %8.2f ms" % (t_ad * 1e3))
print("FD (20 revaluations):               %8.2f ms" % (t_fd * 1e3))
print("speedup: %.1fx" % (t_fd / t_ad))


AD (1 reverse pass, all 10 greeks):    66.44 ms
FD (20 revaluations):                 127.84 ms
speedup: 1.9x


## The gradient is readable Python

Tangent's output is source code, not an opaque tape. Here is the head of the
generated adjoint — ordinary NumPy with an explicit reverse loop over the time
steps, `tangent.unbroadcast` for the shape bookkeeping, and comments mapping
each line back to the primal.

In [7]:
src = dprice.__tangent_source__
print("\n".join(src.splitlines()[:28]))
print("...")
print("(%d lines of generated gradient code you can read, diff, and edit)" % len(src.splitlines()))


def dcaplet_pricedF0sigma(F0, sigma, Z, K, tau, dt, L, e0, M, N, n_steps, b_return=1.0):
    # Initialize the tape
    _stack = tangent.Stack()
    _F = None
    _F2 = None
    drift_times_dt = None
    _F3 = None
    _F4 = None
    drift = None
    _drift = None
    g_times_L = None
    _drift2 = None
    _drift3 = None
    g = None
    _g = None
    tau_times_sigma = None
    _g2 = None
    tau_times_F = None
    # Beginning of forward pass
    'Stylized spot-measure LMM price of a caplet on the first forward rate.\n\n    Differentiated w.r.t. F0 (initial forward curve -> deltas) and sigma\n    (per-rate vols -> vegas). Z (pre-drawn normals), K, tau, dt, L, e0 and the\n    integer shapes are constants.\n    '
    sqrt_dt = dt ** 0.5
    t = (M, N)
    np_ones_t = np.ones(t)
    F = F0 * np_ones_t
    i = 0
    for step in range(n_steps):
        tangent.push(_stack, tau_times_F, '_9662541a')
        tau_times_F = tau * F
...
(226 lines of generated gradient code you can read, diff, a

### Trace any generated line back to your simulator

`tangent.source_map` links each line of the adjoint to the primal statement it
differentiates — so a warning about the backward pass points at *your* code.

In [8]:
seen = set()
for entry in tangent.source_map(dprice, caplet_price):
    p = entry["primal"]
    if p and p not in seen:
        seen.add(p)
        print("primal: %s" % p)


primal: _payoff = tau * _payoff2
primal: payoff = tau * (intrinsic * (intrinsic > 0.0)) / (1.0 + tau * F_reset
primal: intrinsic = F_reset - K
primal: F_reset = F @ e0
primal: F = F * np.exp(drift * dt + sigma * (sqrt_dt * Z[step]))
primal: drift = sigma * (g @ L) - 0.5 * sigma * sigma
primal: g = tau * sigma * F / (1.0 + tau * F)
primal: F = F0 * np.ones((M, N))


## Takeaways

- The **plain NumPy** LMM simulator — state-dependent drift, Monte-Carlo loop —
  was differentiated **as written**, no rewrite into a framework.
- **All greeks in one reverse pass**, matching bump-and-revalue but exact and
  far cheaper as the number of sensitivities grows.
- The gradient is **readable source code** you can inspect with
  `source_map` / `tangent.explain` and even edit with `tangent.insert_grad_of`
  (see the *Gradient Surgery* notebook) — the debuggability a tracing framework
  like the JAX-based `kooderive` cannot offer.

Tangent's niche isn't replacing JAX for large tensor programs; it is making the
derivatives of gnarly, loop-and-branch-heavy scientific and quantitative code
*legible*.
